# 1D Cutting Stock Problem — Formulation and Solution (Julia)

> **ORKit Free** · Mixed-Integer Programming (MIP) · JuMP + HiGHS

---

## The Problem

A factory has master rolls of width $W$. Customers need smaller pieces of different widths.
We must cut the master rolls to satisfy all demands **using as few rolls as possible**.

### Formulation

$$\min \quad \sum_{n \in N} y_n$$

$$\text{s.t.} \quad \sum_{n} x_{in} \geq d_i, \quad \forall i \quad \text{(demand)}$$

$$\sum_{i} w_i \, x_{in} \leq W \, y_n, \quad \forall n \quad \text{(capacity)}$$

$$y_n \in \{0,1\}, \quad x_{in} \in \mathbb{Z}_+$$

In [ ]:
using JuMP
using HiGHS
using JSON3

println("JuMP version: ", pkgversion(JuMP))
println("HiGHS version: ", pkgversion(HiGHS))

## Loading the Instance

In [ ]:
data = JSON3.read(read("../instances/small_3.json", String))

W = data.master_roll
items = data.items
ids = [it.id for it in items]
w = Dict(it.id => it.width for it in items)
d = Dict(it.id => it.demand for it in items)
n_max = sum(it.demand for it in items)

println("Master roll width: $W")
println("Item types: $(length(items))")
println("Upper bound on rolls: $n_max")
for it in items
    println("  Item $(it.id): width=$(it.width), demand=$(it.demand)")
end

## Building the JuMP Model

JuMP provides a clean algebraic interface. We define variables, objective, and constraints
using macros (`@variable`, `@objective`, `@constraint`).

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)

# ── Decision Variables ────────────────────────────
@variable(model, y[1:n_max], Bin)                    # roll n opened?
@variable(model, x[ids, 1:n_max] >= 0, Int)          # pieces of type i from roll n

# ── Objective: minimize rolls ─────────────────────
@objective(model, Min, sum(y))

# ── Demand constraints ────────────────────────────
for i in ids
    @constraint(model, sum(x[i, n] for n = 1:n_max) >= d[i])
end

# ── Capacity constraints ──────────────────────────
for n = 1:n_max
    @constraint(model, sum(w[i] * x[i, n] for i in ids) <= W * y[n])
end

println("Model built!")
println("  Variables: $(num_variables(model))")
println("  Constraints: $(num_constraints(model; count_variable_in_set_constraints=false))")

## Solving

In [ ]:
optimize!(model)

status = termination_status(model)
rolls_used = Int(round(objective_value(model)))

println("Status: $status")
println("Rolls used: $rolls_used")

## Analyzing the Cutting Patterns

In [ ]:
println("Cutting Patterns")
println("=" ^ 50)

total_waste = 0.0
for n = 1:n_max
    if value(y[n]) > 0.5
        cuts = Dict{Int,Int}()
        used_width = 0.0
        for i in ids
            qty = Int(round(value(x[i, n])))
            if qty > 0
                cuts[i] = qty
                used_width += w[i] * qty
            end
        end
        waste = W - used_width
        total_waste += waste
        println("  Roll $n: $cuts  (used=$used_width, waste=$waste)")
    end
end

println()
println("Total waste: $total_waste ($(round(total_waste / (rolls_used * W) * 100; digits=1))%)")

## Demand Verification

In [ ]:
println(rpad("Item", 8), rpad("Width", 8), rpad("Demand", 8), rpad("Cut", 8), "OK?")
println("-" ^ 40)
for it in items
    i = it.id
    total_cut = sum(Int(round(value(x[i, n]))) for n = 1:n_max)
    ok = total_cut >= d[i] ? "YES" : "NO"
    println(rpad(i, 8), rpad(w[i], 8), rpad(d[i], 8), rpad(total_cut, 8), ok)
end

## Key Takeaways

- **JuMP** provides a natural mathematical syntax for optimization models
- **HiGHS** solves MIP problems efficiently as an open-source solver
- The compact formulation works well for small instances but grows quickly
- For large-scale instances, **column generation** (Master bundle) is the standard approach

### References

- Kantorovich, L. V. (1960). Mathematical Methods of Organising and Planning Production.
- Gilmore, P. C., Gomory, R. E. (1961). A linear programming approach to the cutting stock problem.